In [1]:
import polars as pl


In [2]:
ds = pl.scan_parquet("/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/batched_nmf/all_patterns_partitioned")


In [3]:
df_nonzero = ds.filter(pl.col("loading") > 0).collect()

print(df_nonzero)

shape: (211_093_586, 8)
┌────────┬─────────┬────────┬──────────┬──────┬───────────┬─────┬─────┐
│ run_id ┆ pattern ┆ gene   ┆ loading  ┆ seed ┆ tol       ┆ L1  ┆ k   │
│ ---    ┆ ---     ┆ ---    ┆ ---      ┆ ---  ┆ ---       ┆ --- ┆ --- │
│ i32    ┆ i32     ┆ str    ┆ f64      ┆ i32  ┆ f64       ┆ f64 ┆ i64 │
╞════════╪═════════╪════════╪══════════╪══════╪═══════════╪═════╪═════╡
│ 1      ┆ 1       ┆ FBXL18 ┆ 0.000058 ┆ 42   ┆ 0.00001   ┆ 0.1 ┆ 10  │
│ 1      ┆ 2       ┆ FBXL18 ┆ 0.000039 ┆ 42   ┆ 0.00001   ┆ 0.1 ┆ 10  │
│ 1      ┆ 3       ┆ FBXL18 ┆ 0.000075 ┆ 42   ┆ 0.00001   ┆ 0.1 ┆ 10  │
│ 1      ┆ 4       ┆ FBXL18 ┆ 0.000029 ┆ 42   ┆ 0.00001   ┆ 0.1 ┆ 10  │
│ 1      ┆ 5       ┆ FBXL18 ┆ 0.000063 ┆ 42   ┆ 0.00001   ┆ 0.1 ┆ 10  │
│ …      ┆ …       ┆ …      ┆ …        ┆ …    ┆ …         ┆ …   ┆ …   │
│ 749    ┆ 53      ┆ DHRSX  ┆ 0.000061 ┆ 2025 ┆ 0.0000001 ┆ 0.7 ┆ 90  │
│ 749    ┆ 59      ┆ DHRSX  ┆ 0.000046 ┆ 2025 ┆ 0.0000001 ┆ 0.7 ┆ 90  │
│ 749    ┆ 67      ┆ DHRSX  ┆ 0.000276 ┆

In [4]:
unique_genes = df_nonzero.select(pl.col("gene").unique())
print(f"Number of unique genes with non-zero loadings: {unique_genes.height}")
print(unique_genes)

Number of unique genes with non-zero loadings: 34978
shape: (34_978, 1)
┌────────────┐
│ gene       │
│ ---        │
│ str        │
╞════════════╡
│ PXN        │
│ AL390198.1 │
│ IL1B       │
│ TNNI2      │
│ AC090796.1 │
│ …          │
│ GET3       │
│ AC136624.2 │
│ AJ003147.1 │
│ AC132872.1 │
│ SMC2-AS1   │
└────────────┘


In [5]:
percentage_runs_intermediate = (
    df_nonzero
    .group_by(["gene", "k"])
    .agg(
        n_runs_with_gene = pl.col("run_id").n_unique()
    )
    .join(
        df_nonzero.select(["k", "run_id"]).unique()
        .group_by("k")
        .agg(total_runs_for_k = pl.col("run_id").n_unique()),
        on="k"
    )
    .with_columns(
        pct_runs_with_gene = (pl.col("n_runs_with_gene") 
                              / pl.col("total_runs_for_k"))
    )
)
print(percentage_runs_intermediate)


shape: (349_780, 5)
┌────────────┬─────┬──────────────────┬──────────────────┬────────────────────┐
│ gene       ┆ k   ┆ n_runs_with_gene ┆ total_runs_for_k ┆ pct_runs_with_gene │
│ ---        ┆ --- ┆ ---              ┆ ---              ┆ ---                │
│ str        ┆ i64 ┆ u32              ┆ u32              ┆ f64                │
╞════════════╪═════╪══════════════════╪══════════════════╪════════════════════╡
│ METTL14-DT ┆ 40  ┆ 15               ┆ 75               ┆ 0.2                │
│ CLPB       ┆ 30  ┆ 45               ┆ 75               ┆ 0.6                │
│ AL139379.1 ┆ 30  ┆ 30               ┆ 75               ┆ 0.4                │
│ SDK2       ┆ 50  ┆ 75               ┆ 75               ┆ 1.0                │
│ TNFRSF14   ┆ 90  ┆ 45               ┆ 75               ┆ 0.6                │
│ …          ┆ …   ┆ …                ┆ …                ┆ …                  │
│ AC012508.1 ┆ 40  ┆ 15               ┆ 75               ┆ 0.2                │
│ SMIM28     ┆ 90  ┆

In [6]:

genes_percentage_runs_df = (
    percentage_runs_intermediate
    .with_columns(
        pct_col = (pl.lit("pct_k") + pl.col("k").cast(pl.Utf8))
    )
    .pivot(
        index="gene",
        on="pct_col",
        values="pct_runs_with_gene",
        aggregate_function="first",
    )
    .with_columns(
       mean_pct = pl.mean_horizontal(pl.all().exclude("gene")),
    )
    .sort("mean_pct", descending=True)
)
print(genes_percentage_runs_df)

shape: (34_978, 12)
┌────────────┬─────────┬─────────┬─────────┬───┬──────────┬─────────┬─────────┬──────────┐
│ gene       ┆ pct_k40 ┆ pct_k30 ┆ pct_k50 ┆ … ┆ pct_k100 ┆ pct_k70 ┆ pct_k80 ┆ mean_pct │
│ ---        ┆ ---     ┆ ---     ┆ ---     ┆   ┆ ---      ┆ ---     ┆ ---     ┆ ---      │
│ str        ┆ f64     ┆ f64     ┆ f64     ┆   ┆ f64      ┆ f64     ┆ f64     ┆ f64      │
╞════════════╪═════════╪═════════╪═════════╪═══╪══════════╪═════════╪═════════╪══════════╡
│ HSPB1      ┆ 1.0     ┆ 1.0     ┆ 1.0     ┆ … ┆ 1.0      ┆ 1.0     ┆ 1.0     ┆ 1.0      │
│ WWC2       ┆ 1.0     ┆ 1.0     ┆ 1.0     ┆ … ┆ 1.0      ┆ 1.0     ┆ 1.0     ┆ 1.0      │
│ ARNT2      ┆ 1.0     ┆ 1.0     ┆ 1.0     ┆ … ┆ 1.0      ┆ 1.0     ┆ 1.0     ┆ 1.0      │
│ TRIP12     ┆ 1.0     ┆ 1.0     ┆ 1.0     ┆ … ┆ 1.0      ┆ 1.0     ┆ 1.0     ┆ 1.0      │
│ FAM49B     ┆ 1.0     ┆ 1.0     ┆ 1.0     ┆ … ┆ 1.0      ┆ 1.0     ┆ 1.0     ┆ 1.0      │
│ …          ┆ …       ┆ …       ┆ …       ┆ … ┆ …        ┆ …       ┆ 

In [7]:
test_df = ds.filter((pl.col("gene") == "ENO2") & (pl.col("loading") > 0)).collect()
print(test_df)

shape: (22_330, 8)
┌────────┬─────────┬──────┬──────────┬──────┬───────────┬─────┬─────┐
│ run_id ┆ pattern ┆ gene ┆ loading  ┆ seed ┆ tol       ┆ L1  ┆ k   │
│ ---    ┆ ---     ┆ ---  ┆ ---      ┆ ---  ┆ ---       ┆ --- ┆ --- │
│ i32    ┆ i32     ┆ str  ┆ f64      ┆ i32  ┆ f64       ┆ f64 ┆ i64 │
╞════════╪═════════╪══════╪══════════╪══════╪═══════════╪═════╪═════╡
│ 1      ┆ 1       ┆ ENO2 ┆ 0.000061 ┆ 42   ┆ 0.00001   ┆ 0.1 ┆ 10  │
│ 1      ┆ 2       ┆ ENO2 ┆ 0.000102 ┆ 42   ┆ 0.00001   ┆ 0.1 ┆ 10  │
│ 1      ┆ 3       ┆ ENO2 ┆ 0.000801 ┆ 42   ┆ 0.00001   ┆ 0.1 ┆ 10  │
│ 1      ┆ 4       ┆ ENO2 ┆ 0.000599 ┆ 42   ┆ 0.00001   ┆ 0.1 ┆ 10  │
│ 1      ┆ 5       ┆ ENO2 ┆ 0.000049 ┆ 42   ┆ 0.00001   ┆ 0.1 ┆ 10  │
│ …      ┆ …       ┆ …    ┆ …        ┆ …    ┆ …         ┆ …   ┆ …   │
│ 749    ┆ 34      ┆ ENO2 ┆ 0.000956 ┆ 2025 ┆ 0.0000001 ┆ 0.7 ┆ 90  │
│ 749    ┆ 37      ┆ ENO2 ┆ 0.014167 ┆ 2025 ┆ 0.0000001 ┆ 0.7 ┆ 90  │
│ 749    ┆ 44      ┆ ENO2 ┆ 0.002476 ┆ 2025 ┆ 0.0000001 ┆ 0.7 ┆ 90  │
│

In [8]:
genes_that_always_show_up = genes_percentage_runs_df.filter(pl.col("mean_pct") == 1.0).select("gene")
print(f"Genes that show up in ALL runs (every param) in at least one pattern: {genes_that_always_show_up.height}")
print(genes_that_always_show_up)

Genes that show up in ALL runs (every param) in at least one pattern: 5474
shape: (5_474, 1)
┌────────┐
│ gene   │
│ ---    │
│ str    │
╞════════╡
│ HSPB1  │
│ WWC2   │
│ ARNT2  │
│ TRIP12 │
│ FAM49B │
│ …      │
│ OTUB1  │
│ ELP4   │
│ CTSD   │
│ SLIT2  │
│ TUBB2A │
└────────┘
